#  Model Evaluation Notebook - PUGS size level

**Description**

This notebook is used for exploring model performance on different PUGS size at instance level.


**Input folder(s)/file(s)**

7 files
- Test image tiles folder (example: `../data/processed/tiles/test`)
- Public Urban Green Spaces (PUGS) ground truth (geojson file) (`dresden_pugs_gt.geojson`)
- Public Urban Green Spaces (PUGS) ground truth (raster) (`dresden_pugs_gt.geotiff`)
- Public Urban Green Spaces (PUGS) ground truth clipped to Dresden boundary (raster) (`clipped_dresden_pugs_gt.geotiff`)
- The best model checkpoint (example: `epoch=47-val_loss=0.18.ckpt`)
- Metrics file (CSV file) (`metrics.csv`)
- Sentinel-2 image stack with PUGS OSM binary mask and SDT (`stacked_sentinel2_dresden.geotiff`)

**Output folder(s)/file(s)**

2 folders and 3 files
- Output folder for prediction result visualization (example: `../reports/figure/gt_vs_pred/v0`)
- `fn_fp_maps` folder which contains the False Negative (FN) and False Positive (FP) area from each model prediction output
- Loss graph over Epochs (example: `loss_graph_v0.png`)
- Whole area prediction (example: `pred_v0.geotiff`)
- Whole area prediction (clipped to Dresden boundary) (example: `clipped_pred_v0.geotiff`)

**Requirements/Dependencies**

This notebook requires user to run the following notebooks first:
- 1_data_acquisition_osm.ipynb
- 1_data_acquisition_ground_truth.ipynb
- 1_data_acquisition_satellite_image.ipynb
- 2_data_processing_osm.ipynb
- 2_data_processing_ground_truth.ipynb
- 3_data_processing_satellite_image.ipynb
- 5_ground_truth_creation.ipynb
- 6_model_training.ipynb

**Note:** It does not require user to run `4_ground_truth_exploration.ipynb` notebook. The exploration notebook is created for understanding how each ground truth dataset complement each other and help author decide on which datasets will be used to create a final ground truth dataset.

In case all the input data required in this notebook are available, user can directly run this notebook without running the notebooks in the list.

**Remarks**

The output shown in the notebook is run from this model experiment.

- **Input**: Sentinel-2 image and PUGS binary mask derived from OSM (input channels is 14 channels)
- **Version**: 1

More detail about model experiment setup can be found in [Model experiment setup](../models/README.md#model-experiment-setup)

# 1. Import libraries

In [1]:
import os
import sys

# Add the root directory to Python path so that we can import the modules
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root_dir)

In [2]:
import yaml
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from rasterstats import zonal_stats
from pugs_detection.ground_truth import merge_overlapping_polygons
from pugs_detection.plots import plot_result_analysis

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import label as cc_label

from pugs_detection.utils import set_all_seeds, get_checkpoint_path

In [3]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="You will likely lose important projection information when converting to a PROJ string from another format.*",
    category=UserWarning,
)


# 3. Set variables

In [4]:
# Load config
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Set seeds for reproducibility
seed = config["general"]["seed"]
set_all_seeds(seed)

Seed set to 42


In [5]:
gt_path = config["paths"]["gt_undissolved_path"]
osm_pred_raster_path = config["paths"]["with_osm_pred_raster_path"].format(seed=seed)

print("osm_pred_raster_path:", osm_pred_raster_path)
print("gt_path:", gt_path)

osm_pred_raster_path: ../s42/results/clipped_prediction/clipped_pred_v1.geotiff
gt_path: ../data/processed/ground truth/dresden_pugs_gt_undissolved.geojson


In [6]:
"""
Size-stratified segmentation metrics using ground-truth VECTOR polygons,
binned by object area in hectares. 
Each GT polygon's footprint is rasterized individually.

Bins (ha): [0, 0.4], (0.4, 3], (3, 10], (10, 80], (80, inf)

Produces, per bin:
  - Recall (pixel-level, within each GT object's own footprint)
  - Instance-averaged IoU (mean IoU per object, not weighted by area --
    this is what removes the area-weighting bias of plain aggregate IoU)

Requires: geopandas, rasterio, numpy, scipy
    pip install geopandas rasterio scipy --break-system-packages
"""



# ---- CONFIG ---------------------------------------------------------------
BIN_EDGES_HA = [0, 0.4, 3, 10, 80, np.inf]
BIN_LABELS = ["[0, 0.4]", "(0.4, 3]", "(3, 10]", "(10, 80]", "(80, inf)"]
COV_THR= 0.5
COV_THR_03 = 0.3
# -----------------------------------------------------------------------------


def assign_bin(area_ha):
    idx = np.digitize(area_ha, BIN_EDGES_HA[1:-1])
    return BIN_LABELS[idx]


def load_prediction(pred_path):
    """Load a predicted binary raster and its affine transform/CRS."""
    with rasterio.open(pred_path) as src:
        pred = src.read(1)
        transform = src.transform
        crs = src.crs
        shape = (src.height, src.width)
    return (pred > 0).astype(np.uint8), transform, crs, shape


def evaluate(pred_mask, transform, crs, shape, gt_vector_path, pred_structure=None):
    """
    pred_mask: binary numpy array (H, W)
    transform, crs, shape: from the prediction raster (defines the grid
        everything is rasterized onto)
    gt_vector_path: path to GT polygons (shapefile/GeoJSON/GeoPackage)
    pred_structure: connectivity for labeling predicted instances
        (None=4-connectivity, np.ones((3,3))=8-connectivity)
 
    Returns a dict: per-bin {n_objects, tp, fn, recall, mean_instance_iou,
    n_detected, n_missed, detection_rate, detected_object_ids, missed_object_ids},
    plus unmatched_fp_px and overall_precision.
 
    An object is "detected" if its best-overlapping predicted blob has
    IoU > DETECTION_IOU_THRESHOLD; otherwise it's "missed", regardless of
    whether some pixels overlapped (a sliver of overlap isn't a detection).
    """
    gt = gpd.read_file(gt_vector_path)
    if gt.crs != crs:
        gt = gt.to_crs(crs)  # reproject GT to match the prediction raster's CRS
 
    # area() is only meaningful in m^2 for a projected (metric) CRS.
    if crs.is_geographic:
        raise ValueError(
            "CRS is geographic (degrees) -- reproject to a projected CRS "
            "(meters) before computing areas, e.g. gt.estimate_utm_crs()."
        )
 
    pred_bool = pred_mask.astype(bool)
    pred_labels, n_pred = cc_label(pred_bool, pred_structure)
 
    results = {
        b: {"tp": 0, "fn": 0, "n_objects": 0, "ious": [],
            "detected_ids": [], "missed_ids": []}
        for b in BIN_LABELS
    }
    gt_union_mask = np.zeros(shape, dtype=bool)
 
    # use the GT GeoDataFrame's own index as a stable object id so you can
    # trace which specific features were detected/missed back to the vector file
    for obj_id, row in gt.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        area_ha = geom.area / 10_000
        b = assign_bin(area_ha)
 
        obj_mask = rasterize(
            [(geom, 1)], out_shape=shape, transform=transform,
            fill=0, dtype=np.uint8
        ).astype(bool)
 
        # pixel-level recall bookkeeping
        tp = np.logical_and(pred_bool, obj_mask).sum()
        fn = obj_mask.sum() - tp
        results[b]["tp"] += int(tp)
        results[b]["fn"] += int(fn)
        results[b]["n_objects"] += 1
        gt_union_mask |= obj_mask
 
        # instance-averaged IoU: best-overlapping predicted instance, or 0
        # if no predicted instance overlaps this GT object at all
        overlapping_ids = np.unique(pred_labels[obj_mask])
        overlapping_ids = overlapping_ids[overlapping_ids != 0]
        best_iou = 0.0
        for pid in overlapping_ids:
            pred_obj = pred_labels == pid
            inter = np.logical_and(obj_mask, pred_obj).sum()
            union = np.logical_or(obj_mask, pred_obj).sum()
            best_iou = max(best_iou, inter / union if union > 0 else 0.0)
        results[b]["ious"].append(best_iou)
 
        if best_iou > COV_THR:
            results[b]["detected_ids"].append(obj_id)
        else:
            results[b]["missed_ids"].append(obj_id)
 
    # FP: predicted-positive pixels outside every GT polygon
    fp_mask = np.logical_and(pred_bool, ~gt_union_mask)
    unmatched_fp = int(fp_mask.sum())
 
    for b in BIN_LABELS:
        tp, fn = results[b]["tp"], results[b]["fn"]
        results[b]["recall"] = tp / (tp + fn) if (tp + fn) > 0 else None
        ious = results[b].pop("ious")
        results[b]["mean_instance_iou"] = float(np.mean(ious)) if ious else None
 
        n_detected = len(results[b]["detected_ids"])
        n_missed = len(results[b]["missed_ids"])
        results[b]["n_detected"] = n_detected
        results[b]["n_missed"] = n_missed
        results[b]["detection_rate"] = (
            n_detected / (n_detected + n_missed) if (n_detected + n_missed) > 0 else None
        )
        # object ids (GT GeoDataFrame index) kept under
        # results[b]["detected_ids"] / ["missed_ids"] for tracing back to the vector file
 
    results["unmatched_fp_px"] = unmatched_fp
    total_tp = sum(results[b]["tp"] for b in BIN_LABELS)
    results["overall_precision"] = (
        total_tp / (total_tp + unmatched_fp) if (total_tp + unmatched_fp) > 0 else None
    )
    return results

def print_report(results, list_ids=False):
    header = (f"{'Size bin (ha)':<15} {'# objects':>10} {'Detected':>9} {'Missed':>7} "
              f"{'Det.rate':>9} {'Recall':>8} {'Mean IoU':>9}")
    print(header)
    for b in BIN_LABELS:
        r = results[b]
        recall_str = f"{r['recall']:.3f}" if r["recall"] is not None else "n/a"
        iou_str = f"{r['mean_instance_iou']:.3f}" if r["mean_instance_iou"] is not None else "n/a"
        det_rate_str = f"{r['detection_rate']:.3f}" if r["detection_rate"] is not None else "n/a"
        print(f"{b:<15} {r['n_objects']:>10} {r['n_detected']:>9} {r['n_missed']:>7} "
              f"{det_rate_str:>9} {recall_str:>8} {iou_str:>9}")
        if list_ids and r["missed_ids"]:
            print(f"    missed object ids: {r['missed_ids']}")
 
    print(f"\nUnmatched FP pixels (outside all GT polygons): {results['unmatched_fp_px']}")
    if results["overall_precision"] is not None:
        print(f"Overall precision (TP_total / (TP_total+FP_total)): {results['overall_precision']:.3f}")
    print("\nNotes:")
    print(f"- An object is 'Detected' if its best-overlapping predicted blob has")
    print(f"  IoU > {COV_THR}; otherwise 'Missed', regardless of partial pixel overlap.")
    print("- Recall is pixel-level, computed within each GT object's own footprint.")
    print("- Mean IoU is instance-averaged: one IoU value per GT object (vs. its")
    print("  best-overlapping predicted blob, 0 if none), averaged unweighted")
    print("  within each bin -- this avoids large objects dominating the score.")
    print("- False positives have no GT polygon to inherit a size bin from, so")
    print("  they're reported once as an overall figure rather than per bin.")
    print("- detected_ids / missed_ids in the results dict hold the GT GeoDataFrame")
    print("  index for each object, so you can trace specific misses back to the")
    print("  vector file (e.g. gt.loc[missed_ids] to inspect/plot them).")
    print("  they're reported once as an overall figure rather than per bin.")


In [7]:

if __name__ == "__main__":
    # Example:
    pred_mask, transform, crs, shape = load_prediction(osm_pred_raster_path)
    results = evaluate(pred_mask, transform, crs, shape, gt_path)
    print("Results for prediction raster:", osm_pred_raster_path)
    print_report(results,list_ids=False)

Results for prediction raster: ../s42/results/clipped_prediction/clipped_pred_v1.geotiff
Size bin (ha)    # objects  Detected  Missed  Det.rate   Recall  Mean IoU
[0, 0.4]               689        29     660     0.042    0.318     0.076
(0.4, 3]              1144       214     930     0.187    0.633     0.224
(3, 10]                159        48     111     0.302    0.867     0.321
(10, 80]                88        17      71     0.193    0.952     0.265
(80, inf)                8         1       7     0.125    0.989     0.291

Unmatched FP pixels (outside all GT polygons): 46435
Overall precision (TP_total / (TP_total+FP_total)): 0.949

Notes:
- An object is 'Detected' if its best-overlapping predicted blob has
  IoU > 0.5; otherwise 'Missed', regardless of partial pixel overlap.
- Recall is pixel-level, computed within each GT object's own footprint.
- Mean IoU is instance-averaged: one IoU value per GT object (vs. its
  best-overlapping predicted blob, 0 if none), averaged unweighte

In [8]:
# prediction using only sentinel

pred_mask, transform, crs, shape = load_prediction(osm_pred_raster_path)
results = evaluate(pred_mask, transform, crs, shape, gt_path)
print("Results for prediction raster:", osm_pred_raster_path)
print_report(results,list_ids=False)

Results for prediction raster: ../s42/results/clipped_prediction/clipped_pred_v1.geotiff
Size bin (ha)    # objects  Detected  Missed  Det.rate   Recall  Mean IoU
[0, 0.4]               689        29     660     0.042    0.318     0.076
(0.4, 3]              1144       214     930     0.187    0.633     0.224
(3, 10]                159        48     111     0.302    0.867     0.321
(10, 80]                88        17      71     0.193    0.952     0.265
(80, inf)                8         1       7     0.125    0.989     0.291

Unmatched FP pixels (outside all GT polygons): 46435
Overall precision (TP_total / (TP_total+FP_total)): 0.949

Notes:
- An object is 'Detected' if its best-overlapping predicted blob has
  IoU > 0.5; otherwise 'Missed', regardless of partial pixel overlap.
- Recall is pixel-level, computed within each GT object's own footprint.
- Mean IoU is instance-averaged: one IoU value per GT object (vs. its
  best-overlapping predicted blob, 0 if none), averaged unweighte

In [9]:

# Read the undissolved dataset - it was used to create GT binary mask
undissolved_path = config["paths"]["gt_undissolved_path"]
undissolved_gdf = gpd.read_file(undissolved_path)
# # to inspect/plot the missed objects in a given bin:
missed = undissolved_gdf.loc[results["[0, 0.4]"]["missed_ids"]]


In [11]:
import psutil
print(f"{psutil.virtual_memory().percent}% RAM used")

82.7% RAM used


In [12]:
import sys
for name, obj in list(globals().items()):
    size = sys.getsizeof(obj)
    if size > 1_000_000:  # >1MB
        print(name, type(obj), size)

pred_mask <class 'numpy.ndarray'> 6140096
undissolved_gdf <class 'geopandas.geodataframe.GeoDataFrame'> 1106987
